
📊 aula_teste — Integração de Dados e ETL

**Atividade prática com lacunas progressivas**

Neste notebook você vai construir um pipeline ETL **passo a passo**, preenchendo os espaços em branco.

---

### 🗺️ Como funciona
| Nível | Símbolo | O que fazer |
|---|---|---|
| Fácil | `___` | Preencher uma palavra ou valor simples |
| Médio | `# ??? ...` | Escrever uma linha de código |
| Difícil | `# 🏆 DESAFIO` | Construir o bloco inteiro |

> ✅ **Regra:** Execute cada célula com **Shift + Enter** após preencher. Se der erro, releia o comentário acima da lacuna — ele sempre dá uma dica!
---



---
## 🔧 Etapa 0 — Setup

Execute esta célula para preparar o ambiente. Não precisa alterar nada aqui.

In [2]:
# Importações — execute sem alterar
import pandas as pd
import sqlite3
import random
from pathlib import Path
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

print("✅ Ambiente pronto!")
print(f"   Pandas: {pd.__version__}")

✅ Ambiente pronto!
   Pandas: 2.2.2


---
## 🏗️ Etapa 1 — Criação dos Dados de Exemplo

Execute as células abaixo para criar as três fontes de dados que vamos usar.
Observe a estrutura de cada uma antes de continuar.


### 1.1 🗄️ Banco de Dados — Tabela `clientes`

In [3]:
# Cria banco SQLite com tabela de clientes — execute sem alterar
conn = sqlite3.connect("loja_teste.db")
conn.executescript("""
    DROP TABLE IF EXISTS clientes;
    CREATE TABLE clientes (
        id INTEGER PRIMARY KEY, nome TEXT, email TEXT,
        cidade TEXT, estado TEXT, segmento TEXT, data_cadastro TEXT
    );
""")
dados = [
    (1,  "Ana Carolina Silva",    "ana@email.com",     "São Paulo",      "SP", "Varejo",      "2022-03-15"),
    (2,  "Bruno Ferreira Lima",   "bruno@gmail.com",   "Rio de Janeiro", "RJ", "Corporativo", "2021-07-22"),
    (3,  "Camila Souza Ramos",    "camila@email.com",  "Belo Horizonte", "MG", "Varejo",      "2023-01-10"),
    (4,  "Daniel Costa",          "daniel@corp.com",   "Curitiba",       "PR", "Corporativo", "2020-11-05"),
    (5,  "Eduarda Mendes",        "edu@email.com",     "Porto Alegre",   "RS", "Varejo",      "2022-09-30"),
    (6,  "Felipe Torres",         "felipe@corp.com",   "Salvador",       "BA", "Corporativo", "2021-04-18"),
    (7,  "Gabriela Pinto",        "gabi@gmail.com",    "Fortaleza",      "CE", "Varejo",      "2023-06-02"),
    (8,  "Henrique Alves",        "henrique@corp.com", "Recife",         "PE", "Corporativo", "2022-12-14"),
    (9,  "Isabela Rocha",         "isa@email.com",     "Goiânia",        "GO", "Varejo",      "2021-08-27"),
    (10, "João Victor Nunes",     None,                "Manaus",         "AM", "Varejo",      "2023-03-19"),
]
conn.executemany("INSERT INTO clientes VALUES (?,?,?,?,?,?,?)", dados)
conn.commit()
print(f"✅ Banco criado com {len(dados)} clientes")
conn.close()

✅ Banco criado com 10 clientes


### 1.2 📄 Arquivo CSV — Pedidos

In [4]:
# Cria arquivo CSV com pedidos — execute sem alterar
import openpyxl
random.seed(42)
pedidos = []
for i in range(1, 31):
    pedidos.append({
        "pedido_id":   i,
        "cliente_id":  random.choice([1,2,3,4,5,6,7,8,9,10]),
        "produto_id":  random.randint(1, 5),
        "quantidade":  random.randint(1, 4),
        "data_pedido": (datetime.now() - timedelta(days=random.randint(1,90))).strftime("%d/%m/%Y"),
        "status":      random.choices(["concluido","em_andamento","cancelado"], weights=[70,20,10])[0],
        "cupom":       random.choice(["PROMO10","","","VOLTA20",""])
    })
# Problemas intencionais para os alunos descobrirem
pedidos.append({"pedido_id": 30, "cliente_id": 5,  "produto_id": 2, "quantidade": 1,  "data_pedido": "10/05/2025", "status": "concluido", "cupom": ""})  # duplicata
pedidos.append({"pedido_id": 31, "cliente_id": 99, "produto_id": 1, "quantidade": 2,  "data_pedido": "01/01/2025", "status": "concluido", "cupom": ""})  # cliente inválido
pedidos.append({"pedido_id": 32, "cliente_id": 3,  "produto_id": 3, "quantidade": -1, "data_pedido": "15/03/2025", "status": "concluido", "cupom": ""})  # qtd inválida

pd.DataFrame(pedidos).to_csv("pedidos_teste.csv", index=False, sep=";")
print(f"✅ CSV criado com {len(pedidos)} linhas (inclui problemas de qualidade!)")

✅ CSV criado com 33 linhas (inclui problemas de qualidade!)


### 1.3 📊 Arquivo Excel — Produtos

In [5]:
# Cria produtos.xlsx — execute sem modificar
import openpyxl
from openpyxl.styles import Font, PatternFill

produtos = {
    "produto_id":     [1, 2, 3, 4, 5, 6, 7, 8],
    "nome":           ["Notebook Pro 15","Mouse Sem Fio","Teclado Mecanico",
                       "Monitor 24pol","Headset USB","Webcam Full HD","SSD 1TB","Hub USB-C"],
    "categoria":      ["Computadores","Perifericos","Perifericos","Monitores",
                       "Audio","Perifericos","Armazenamento","Acessorios"],
    "preco_unitario": [3499.90, 129.90, 349.90, 1199.90, 249.90, 399.90, 499.90, 189.90],
    "custo":          [2100.00, 55.00, 180.00, 620.00, 95.00, 160.00, 230.00, 78.00],
    "estoque":        [45, 230, 88, 52, 175, 63, 120, 200],
    "ativo":          ["Sim","Sim","Sim","Sim","Sim","Nao","Sim","Sim"],
}
df_prod = pd.DataFrame(produtos)
with pd.ExcelWriter("produtos.xlsx", engine="openpyxl") as writer:
    df_prod.to_excel(writer, sheet_name="Catalogo", index=False)
    ws = writer.sheets["Catalogo"]
    for cell in ws[1]:
        cell.fill = PatternFill("solid", fgColor="1E3A5F")
        cell.font = Font(color="FFFFFF", bold=True)
    for i, row in enumerate(ws.iter_rows(min_row=2, max_row=ws.max_row), 2):
        for cell in row:
            cell.fill = PatternFill("solid", fgColor="EFF6FF" if i%2==0 else "FFFFFF")
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = max(len(str(c.value or "")) for c in col)+4

print(f"Arquivo 'produtos.xlsx' criado: {len(df_prod)} produtos")
df_prod


Arquivo 'produtos.xlsx' criado: 8 produtos


,produto_id,nome,categoria,preco_unitario,custo,estoque,ativo
0,1,Notebook Pro 15,Computadores,3499.90,2100.00,45,Sim
1,2,Mouse Sem Fio,Perifericos,129.90,55.00,230,Sim
2,3,Teclado Mecanico,Perifericos,349.90,180.00,88,Sim
3,4,Monitor 24pol,Monitores,1199.90,620.00,52,Sim
4,5,Headset USB,Audio,249.90,95.00,175,Sim
5,6,Webcam Full HD,Perifericos,399.90,160.00,63,Nao
6,7,SSD 1TB,Armazenamento,499.90,230.00,120,Sim
7,8,Hub USB-C,Acessorios,189.90,78.00,200,Sim


---
## 📤 Etapa 2 — EXTRACT (Extração)

> **Objetivo:** ler os dados brutos de cada fonte sem transformar nada ainda.

### 🟢 Nível Fácil — complete as lacunas `___`

### 2.1 Extraindo do Banco de Dados

In [6]:
# ── LACUNA 1 (Fácil) ──────────────────────────────────────
# SQLAlchemy precisa de uma string de conexão.
# Para SQLite local, o formato é: "sqlite:///nome_do_arquivo.db"
# Preencha o nome correto do banco que criamos acima:

engine = create_engine("sqlite:///loja_teste.db")   # ← preencha aqui

# Agora leia a tabela 'clientes' usando pd.read_sql()
# Dica: pd.read_sql("SELECT * FROM nome_da_tabela", conexão)

df_clientes = pd.read_sql("SELECT * FROM clientes", engine)  # ← preencha a tabela

print(f"✅ Clientes extraídos: {len(df_clientes)} registros")
df_clientes.head()

✅ Clientes extraídos: 10 registros


,id,nome,email,cidade,estado,segmento,data_cadastro
0,1,Ana Carolina Silva,ana@email.com,São Paulo,SP,Varejo,2022-03-15
1,2,Bruno Ferreira Lima,bruno@gmail.com,Rio de Janeiro,RJ,Corporativo,2021-07-22
2,3,Camila Souza Ramos,camila@email.com,Belo Horizonte,MG,Varejo,2023-01-10
3,4,Daniel Costa,daniel@corp.com,Curitiba,PR,Corporativo,2020-11-05
4,5,Eduarda Mendes,edu@email.com,Porto Alegre,RS,Varejo,2022-09-30


### 2.2 Extraindo do CSV

In [7]:
# ── LACUNA 2 (Fácil) ──────────────────────────────────────
# O CSV usa ponto-e-vírgula (;) como separador.
# pd.read_csv() tem um parâmetro chamado 'sep' para isso.

df_pedidos = pd.read_csv(
    "pedidos_teste.csv",
    sep=";",           # ← qual separador usar? (string)
    encoding="utf-8"
)

print(f"✅ Pedidos extraídos: {len(df_pedidos)} registros")

# Confira a distribuição de status:
print("\nDistribuição de status:")
print(df_pedidos["status"].value_counts())  # ← qual coluna mostrar?

✅ Pedidos extraídos: 33 registros

Distribuição de status:
status
concluido       25
em_andamento     7
cancelado        1
Name: count, dtype: int64


### 2.3 Extraindo do Excel

In [8]:
# ── LACUNA 3 (Fácil) ──────────────────────────────────────
# pd.read_excel() precisa do nome da aba (sheet_name)
# e da engine para .xlsx (openpyxl).
# A aba do nosso arquivo se chama "Catalogo".

df_produtos = pd.read_excel(
    "produtos.xlsx",
    sheet_name="Catalogo",     # ← nome da aba (string)
    engine="openpyxl"          # ← engine para xlsx (string)
)

print(f"✅ Produtos extraídos: {len(df_produtos)} registros")
df_produtos

✅ Produtos extraídos: 8 registros


,produto_id,nome,categoria,preco_unitario,custo,estoque,ativo
0,1,Notebook Pro 15,Computadores,3499.90,2100,45,Sim
1,2,Mouse Sem Fio,Perifericos,129.90,55,230,Sim
2,3,Teclado Mecanico,Perifericos,349.90,180,88,Sim
3,4,Monitor 24pol,Monitores,1199.90,620,52,Sim
4,5,Headset USB,Audio,249.90,95,175,Sim
5,6,Webcam Full HD,Perifericos,399.90,160,63,Nao
6,7,SSD 1TB,Armazenamento,499.90,230,120,Sim
7,8,Hub USB-C,Acessorios,189.90,78,200,Sim


### 2.4 Diagnóstico das Fontes Extraídas

In [9]:
# ── LACUNA 4 (Fácil) ──────────────────────────────────────
# O método .isnull().sum() conta os valores nulos de cada coluna.
# Rode o diagnóstico para cada DataFrame:

print("=== DIAGNÓSTICO DE NULOS ===")
print("\n📋 Clientes:")
print(df_clientes.isnull().sum())          # ← método para contar nulos

print("\n📋 Pedidos:")
print(df_pedidos.isnull().sum())  # ← segundo encadeamento

print("\n📋 Produtos:")
print(df_produtos.isnull().sum())

print("\n=== TAMANHO DOS DATASETS ===")
for nome, df in [("Clientes", df_clientes), ("Pedidos", df_pedidos), ("Produtos", df_produtos)]:
    print(f"  {nome}: {df.shape[0]} linhas × {df.shape[1]} colunas")  # ← use df.shape

=== DIAGNÓSTICO DE NULOS ===

📋 Clientes:
id               0
nome             0
email            1
cidade           0
estado           0
segmento         0
data_cadastro    0
dtype: int64

📋 Pedidos:
pedido_id       0
cliente_id      0
produto_id      0
quantidade      0
data_pedido     0
status          0
cupom          22
dtype: int64

📋 Produtos:
produto_id        0
nome              0
categoria         0
preco_unitario    0
custo             0
estoque           0
ativo             0
dtype: int64

=== TAMANHO DOS DATASETS ===
  Clientes: 10 linhas × 7 colunas
  Pedidos: 33 linhas × 7 colunas
  Produtos: 8 linhas × 7 colunas


---
## ⚙️ Etapa 3 — TRANSFORM (Transformação)

> **Objetivo:** limpar, padronizar e enriquecer os dados de cada fonte.

### 🟡 Nível Médio — escreva a linha de código indicada

### 3.1 Transformando Clientes

In [10]:
# ATIVIDADE 4 — Transforme os clientes
df_clientes_t = df_clientes.copy()

# 4a. Padronizar nome (Title Case + sem espaços)
# DICA: .str.title().str.strip()
df_clientes_t["nome"] = df_clientes_t["nome"].str.title().str.strip()  # TODO: complete aqui

# 4b. Preencher emails nulos
# DICA: .fillna("nao_informado@techstore.com")
df_clientes_t["email"] = df_clientes_t["email"].fillna("nao_informado@techstore.com")  # TODO: complete aqui

# 4c. Converter data_cadastro para datetime
# DICA: pd.to_datetime(coluna)
df_clientes_t["data_cadastro"] = pd.to_datetime(df_clientes_t["data_cadastro"])  # TODO: complete aqui

# 4d. Calcular dias como cliente (pronto — só execute)
df_clientes_t["dias_cliente"] = (pd.Timestamp.now() - df_clientes_t["data_cadastro"]).dt.days

print("Clientes transformados:")
print(df_clientes_t[["nome","email","data_cadastro","dias_cliente"]].head(5))

Clientes transformados:
                  nome             email data_cadastro  dias_cliente
0   Ana Carolina Silva     ana@email.com    2022-03-15          1519
1  Bruno Ferreira Lima   bruno@gmail.com    2021-07-22          1755
2   Camila Souza Ramos  camila@email.com    2023-01-10          1218
3         Daniel Costa   daniel@corp.com    2020-11-05          2014
4       Eduarda Mendes     edu@email.com    2022-09-30          1320


### 3.2 Classificando Clientes por Maturidade

In [11]:
# ── LACUNA 6 (Médio) ──────────────────────────────────────
# pd.cut() divide valores numéricos em faixas (bins) e rotula cada faixa.
# Sintaxe: pd.cut(coluna, bins=[limites], labels=["rótulos"])
#
# Crie a coluna "maturidade" com as faixas:
#   0 a 365 dias    → "Novo"
#   365 a 730 dias  → "Intermediário"
#   730+ dias       → "Veterano"

df_clientes_t["maturidade"] = pd.cut(
    df_clientes_t["dias_cliente"],
    bins=[0, 365, 730, float("inf")],   # ← os três limites numéricos
    labels=["Novo", "Intermediário", "Veterano"]           # ← os três rótulos
)

print("Distribuição de maturidade:")
print(df_clientes_t["maturidade"].value_counts())

Distribuição de maturidade:
maturidade
Veterano         10
Novo              0
Intermediário     0
Name: count, dtype: int64


### 3.3 Transformando Pedidos — Diagnóstico e Limpeza

In [12]:
# ── LACUNA 7 (Médio) ──────────────────────────────────────
# Antes de limpar, vamos quantificar os problemas:

n_total = len(df_pedidos)
n_dup   = df_pedidos.duplicated(subset=["pedido_id"]).sum()     # ← qual coluna identifica duplicatas?
n_inv   = (df_pedidos["quantidade"] <= 0).sum()                # ← qual coluna tem valor inválido?
n_orphan= (~df_pedidos["cliente_id"].isin(df_clientes_t["id"])).sum()

print("=== PROBLEMAS ENCONTRADOS ===")
print(f"  Total de linhas         : {n_total}")
print(f"  Duplicatas              : {n_dup}")
print(f"  Quantidade inválida     : {n_inv}")
print(f"  Clientes inexistentes   : {n_orphan}")

=== PROBLEMAS ENCONTRADOS ===
  Total de linhas         : 33
  Duplicatas              : 1
  Quantidade inválida     : 1
  Clientes inexistentes   : 1


In [13]:
# ATIVIDADE 5 — Limpe os pedidos (3 problemas para resolver!)
df_pedidos_t = df_pedidos.copy()
n_total = len(df_pedidos_t)
print(f"ANTES da limpeza: {n_total} linhas")

# 5a. Remover duplicatas (pelo pedido_id, manter a primeira ocorrência)
# DICA: .drop_duplicates(subset="pedido_id", keep="first")
df_pedidos_t = df_pedidos_t.drop_duplicates(subset="pedido_id", keep="first")  # TODO: complete aqui

print(f"  Apos remover duplicatas: {len(df_pedidos_t)} linhas")

# 5b. Remover quantidades invalidas (<= 0)
# DICA: df_pedidos_t[df_pedidos_t["quantidade"] > 0]
df_pedidos_t = df_pedidos_t[df_pedidos_t["quantidade"] > 0]  # TODO: complete aqui

print(f"  Apos remover qtd invalida: {len(df_pedidos_t)} linhas")

# 5c. Remover clientes inexistentes (use .isin())
ids_validos = set(df_clientes_t["id"])
df_pedidos_t = df_pedidos_t[df_pedidos_t["cliente_id"].isin(ids_validos)]  # TODO: complete aqui

print(f"  Apos remover clientes invalidos: {len(df_pedidos_t)} linhas")

# 5d. Converter data_pedido formato dd/mm/yyyy
# DICA: pd.to_datetime(col, format="%d/%m/%Y")
df_pedidos_t["data_pedido"] = pd.to_datetime(df_pedidos_t["data_pedido"], format="%d/%m/%Y")  # TODO: complete aqui

# 5e. Cupom vazio para None (pronto)
df_pedidos_t["cupom"] = df_pedidos_t["cupom"].replace("", None)

print(f"\nPedidos validos finais: {len(df_pedidos_t)}")
df_pedidos_t.head(5)

ANTES da limpeza: 33 linhas
  Apos remover duplicatas: 32 linhas
  Apos remover qtd invalida: 31 linhas
  Apos remover clientes invalidos: 30 linhas

Pedidos validos finais: 30


,pedido_id,cliente_id,produto_id,quantidade,data_pedido,status,cupom
0,1,2,1,3,2026-04-10,concluido,PROMO10
1,2,9,1,4,2026-05-07,concluido,NaN
2,3,4,5,1,2026-03-01,concluido,NaN
3,4,7,2,4,2026-02-25,concluido,PROMO10
4,5,3,4,3,2026-04-06,concluido,NaN


### 3.4 Transformando Produtos

In [14]:
# ATIVIDADE 6 — Transforme os produtos
df_produtos_t = df_produtos.copy()

# 6a. Converter "ativo" de "Sim"/"Nao" para booleano
# DICA: df["col"].str.upper() == "SIM"
df_produtos_t["ativo"] = df_produtos_t["ativo"].str.upper() == "SIM"  # TODO: complete aqui

# 6b. Calcular margem de lucro (preco - custo)
df_produtos_t["margem_lucro"] = df_produtos_t["preco_unitario"] - df_produtos_t["custo"]  # TODO: complete aqui

# 6c. Calcular margem percentual, 2 casas decimais
# DICA: (margem_lucro / preco_unitario * 100).round(2)
df_produtos_t["margem_pct"] = ((df_produtos_t["margem_lucro"] / df_produtos_t["preco_unitario"]) * 100).round(2)  # TODO: complete aqui

print("Produtos transformados:")
print(df_produtos_t[["nome","preco_unitario","custo","margem_lucro","margem_pct","ativo"]])

Produtos transformados:
               nome  preco_unitario  custo  margem_lucro  margem_pct  ativo
0   Notebook Pro 15         3499.90   2100       1399.90       40.00   True
1     Mouse Sem Fio          129.90     55         74.90       57.66   True
2  Teclado Mecanico          349.90    180        169.90       48.56   True
3     Monitor 24pol         1199.90    620        579.90       48.33   True
4       Headset USB          249.90     95        154.90       61.98   True
5    Webcam Full HD          399.90    160        239.90       59.99  False
6           SSD 1TB          499.90    230        269.90       53.99   True
7         Hub USB-C          189.90     78        111.90       58.93   True


---
## 🔗 Etapa 4 — Integração: Merge entre as Fontes

> **Objetivo:** combinar as três fontes em um único dataset analítico.

### 🟠 Nível Médio-Difícil — complete o merge encadeado


In [15]:
# ── LACUNA 10 (Médio-Difícil) ─────────────────────────────
#
# MERGE 1: Pedidos + Clientes
# Queremos trazer nome, cidade, estado, segmento e maturidade
# para cada pedido.
#
# pd.merge(esquerda, direita, left_on="chave_esq", right_on="chave_dir", how="tipo")
# Tipo: "left" (todos os pedidos, mesmo sem cliente)

df_m1 = pd.merge(
    df_pedidos_t,
    df_clientes_t[["id", "nome", "cidade", "estado", "segmento", "maturidade"]],  # ← quais colunas trazer de clientes?
    left_on="cliente_id",      # ← chave em df_pedidos_t
    right_on="id",     # ← chave em df_clientes_t
    how="left"           # ← tipo do join
).drop(columns="id") # remove coluna duplicada

print(f"Após merge 1: {df_m1.shape}")

Após merge 1: (30, 12)


In [16]:
# ── LACUNA 11 (Médio-Difícil) ─────────────────────────────
#
# MERGE 2: (Pedidos+Clientes) + Produtos
# Trazer nome do produto, categoria, preco_unitario e custo.
# A chave é "produto_id" (mesma em ambos os DataFrames).

df_m2 = pd.merge(
    df_m1,                                             # ← DataFrame da esquerda (resultado do merge 1)
    df_produtos_t[["produto_id", "nome", "categoria", "preco_unitario", "custo"]],             # ← quais colunas de produtos?
    on="produto_id",                                          # ← coluna-chave (igual nos dois lados)
    how="left",
    suffixes=("_cliente", "_produto")
).rename(columns={"nome_cliente": "cliente", "nome_produto": "produto"})

print(f"Após merge 2: {df_m2.shape}")
df_m2[["pedido_id","cliente","produto","quantidade","preco_unitario","status"]].head(5)

Após merge 2: (30, 16)


,pedido_id,cliente,produto,quantidade,preco_unitario,status
0,1,Bruno Ferreira Lima,Notebook Pro 15,3,3499.90,concluido
1,2,Isabela Rocha,Notebook Pro 15,4,3499.90,concluido
2,3,Daniel Costa,Headset USB,1,249.90,concluido
3,4,Gabriela Pinto,Mouse Sem Fio,4,129.90,concluido
4,5,Camila Souza Ramos,Monitor 24pol,3,1199.90,concluido


### 4.1 Colunas Calculadas Pós-Merge

In [17]:
# ATIVIDADE 9 — Colunas calculadas pos-merge
df_final = df_m2.copy()

# Mapa de descontos
desconto_map = {"PROMO10": 0.10, "FRETE0": 0.0, "VOLTA20": 0.20}

# 9a. Desconto percentual (use .map() + .fillna(0))
df_final["desconto_pct"] = df_final["cupom"].map(desconto_map).fillna(0)

# 9b. Valor bruto = quantidade x preco_unitario
df_final["valor_bruto"] = df_final["quantidade"] * df_final["preco_unitario"]  # TODO: complete aqui

# 9c. Valor do desconto em reais
df_final["valor_desconto"] = df_final["valor_bruto"] * df_final["desconto_pct"]  # TODO: complete aqui

# 9d. Valor final pago
df_final["valor_final"] = df_final["valor_bruto"] - df_final["valor_desconto"]  # TODO: complete aqui

# 9e. Lucro = valor_final - (quantidade x custo)
df_final["lucro"] = (df_final["valor_final"] - df_final["quantidade"] * df_final["custo"]).round(2)

print("Colunas calculadas criadas!")
df_final[["pedido_id","cliente","produto","valor_bruto","desconto_pct","valor_final","lucro"]].head(8)

Colunas calculadas criadas!


,pedido_id,cliente,produto,valor_bruto,desconto_pct,valor_final,lucro
0,1,Bruno Ferreira Lima,Notebook Pro 15,10499.70,0.10,9449.73,3149.73
1,2,Isabela Rocha,Notebook Pro 15,13999.60,0.00,13999.60,5599.60
2,3,Daniel Costa,Headset USB,249.90,0.00,249.90,154.90
3,4,Gabriela Pinto,Mouse Sem Fio,519.60,0.10,467.64,247.64
4,5,Camila Souza Ramos,Monitor 24pol,3599.70,0.00,3599.70,1739.70
5,6,Bruno Ferreira Lima,Notebook Pro 15,13999.60,0.00,13999.60,5599.60
6,7,João Victor Nunes,Teclado Mecanico,349.90,0.20,279.92,99.92
7,8,Bruno Ferreira Lima,Headset USB,749.70,0.00,749.70,464.70


---
## 🔍 Etapa 5 — Análise dos Dados

> **Objetivo:** explorar o dataset integrado para extrair insights.

### 🟡 Nível Médio — complete as análises


In [18]:
# ATIVIDADE 10 — Filtrar e calcular indicadores
# 10a. Filtre apenas status == "concluido"
df_ok = df_final[df_final["status"] == "concluido"]  # TODO: complete aqui

# 10b. Complete os indicadores:
receita = df_ok["valor_final"].sum()
lucro   = df_ok["lucro"].sum()
n_ped   = len(df_ok)
ticket  = df_ok["valor_final"].mean()

print("=" * 45)
print("  INDICADORES (pedidos concluidos)")
print("=" * 45)
print(f"  Pedidos    : {n_ped}")
print(f"  Receita    : R$ {receita:,.2f}")
print(f"  Lucro      : R$ {lucro:,.2f}")
print(f"  Ticket medio: R$ {ticket:,.2f}")
print(f"  Margem     : {lucro/receita*100:.1f}%")

  INDICADORES (pedidos concluidos)
  Pedidos    : 22
  Receita    : R$ 62,664.37
  Lucro      : R$ 26,219.37
  Ticket medio: R$ 2,848.38
  Margem     : 41.8%


In [19]:
# ── LACUNA 14 (Médio) ─────────────────────────────────────
# Receita por categoria de produto usando groupby + agg

# Dica: .groupby("coluna").agg(alias=("coluna_alvo","função"))
# Funções comuns: "sum", "count", "mean", "nunique"

receita_cat = (df_ok
    .groupby("categoria")                        # ← agrupar por qual coluna?
    .agg(
        pedidos  = ("pedido_id", "count"),   # ← contar pedidos
        receita  = ("valor_final", "sum"), # ← somar receita
        lucro    = ("lucro", "sum")
    )
    .sort_values("receita", ascending=False)   # ← ordenar por receita
    .round(2)
)

print("📊 Receita por Categoria:")
print(receita_cat)

📊 Receita por Categoria:
              pedidos  receita    lucro
categoria                              
Computadores        4 47948.63 18548.63
Monitores           2  7199.40  3479.40
Audio               6  4148.34  2533.34
Perifericos        10  3368.00  1658.00


In [20]:
# ── LACUNA 15 (Médio) ─────────────────────────────────────
# Top 3 clientes por receita

top_clientes = (df_ok
    .groupby(["cliente", "segmento"])           # ← agrupar por cliente e segmento
    .agg(
        pedidos = ("pedido_id","count"),
        receita = ("valor_final","sum")
    )
    .sort_values("receita", ascending=False)  # ← True ou False?
    .head(3)                              # ← quantos mostrar?
    .round(2)
)

print("👑 Top 3 Clientes:")
print(top_clientes)

👑 Top 3 Clientes:
                                 pedidos  receita
cliente             segmento                     
Bruno Ferreira Lima Corporativo        5 25078.63
Isabela Rocha       Varejo             4 24979.93
Camila Souza Ramos  Varejo             3  4960.11


---
## 📥 Etapa 6 — LOAD (Carga no Data Warehouse)

> **Objetivo:** persistir os dados transformados em um banco de destino.

### 🏆 DESAFIO — construa a carga completa

In [21]:
# ── LACUNA 16 (Difícil — Desafio) ────────────────────────
# Crie o Data Warehouse e carregue as tabelas.
# Siga os comentários como guia:

# 1. Crie a engine para o banco de destino "loja_dw.db"
engine_dw = create_engine("sqlite:///loja_dw.db")   # ← string de conexão SQLite

# 2. Carregue a tabela de clientes transformados como "dim_clientes"
#    Dica: df.to_sql(name, con, if_exists, index)
#    Use if_exists="replace" e index=False
dim_clientes = df_clientes_t.copy()
dim_clientes["maturidade"] = dim_clientes["maturidade"].astype(str)
# ??? escreva a linha abaixo:
dim_clientes.to_sql("dim_clientes", engine_dw, if_exists="replace", index=False)

# 3. Carregue a tabela de produtos transformados como "dim_produtos"
dim_produtos = df_produtos_t.copy()
dim_produtos["ativo"] = dim_produtos["ativo"].astype(int)
# ??? escreva a linha abaixo:
dim_produtos.to_sql("dim_produtos", engine_dw, if_exists="replace", index=False)

# 4. Carregue a tabela fato com pedidos concluídos como "fato_vendas"
fato = df_ok.copy()
fato["data_pedido"]  = fato["data_pedido"].dt.strftime("%Y-%m-%d")
fato["maturidade"]   = fato["maturidade"].astype(str)
fato["data_carga"]   = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
# ??? escreva a linha abaixo:
fato.to_sql("fato_vendas", engine_dw, if_exists="replace", index=False)

print("✅ Data Warehouse carregado!")
print("   Tabelas criadas: dim_clientes, dim_produtos, fato_vendas")

✅ Data Warehouse carregado!
   Tabelas criadas: dim_clientes, dim_produtos, fato_vendas


### 6.1 Validação com SQL

In [23]:
# ── LACUNA 17 (Difícil — Desafio) ────────────────────────
# Escreva uma query SQL que:
#   - faça JOIN entre fato_vendas e dim_clientes
#   - agrupe por segmento do cliente
#   - retorne: segmento, total_pedidos, receita_total, lucro_total

query = """
SELECT
    c.segmento,                                    -- coluna de segmento (com alias da tabela)
    COUNT(f.pedido_id)    AS total_pedidos,
    ROUND(SUM(f.valor_final), 2)    AS receita_total, -- coluna de receita
    ROUND(SUM(f.lucro), 2)    AS lucro_total    -- coluna de lucro
FROM fato_vendas f
INNER JOIN dim_clientes c ON f.cliente_id = c.id  -- tipo de join (Corrigido: c.cliente_id para c.id)
GROUP BY c.segmento              -- mesmo campo do SELECT
ORDER BY receita_total DESC
"""

df_sql = pd.read_sql(query, engine_dw)
print("📊 Receita por Segmento (via SQL no DW):")
print(df_sql)

📊 Receita por Segmento (via SQL no DW):
      segmento  total_pedidos  receita_total  lucro_total
0       Varejo             13       32336.70     13741.70
1  Corporativo              9       30327.67     12477.67


---
## ✅ Etapa 7 — Resumo do Seu Pipeline

Execute a célula abaixo para ver o resultado final do seu trabalho!

In [24]:
# ── Célula final — execute após completar todas as lacunas ──
try:
    print("=" * 55)
    print("       RESUMO DO SEU PIPELINE ETL")
    print("=" * 55)

    with engine_dw.connect() as conn:
        n_cli  = conn.execute(text("SELECT COUNT(*) FROM dim_clientes")).scalar()
        n_prod = conn.execute(text("SELECT COUNT(*) FROM dim_produtos")).scalar()
        n_fat  = conn.execute(text("SELECT COUNT(*) FROM fato_vendas")).scalar()
        rec    = conn.execute(text("SELECT ROUND(SUM(valor_final),2) FROM fato_vendas")).scalar()
        luc    = conn.execute(text("SELECT ROUND(SUM(lucro),2) FROM fato_vendas")).scalar()

    print(f"\n📥 Fontes extraídas:")
    print(f"   Banco SQLite  → {len(df_clientes)} clientes")
    print(f"   CSV           → {len(df_pedidos)} pedidos brutos")
    print(f"   Excel         → {len(df_produtos)} produtos")

    print(f"\n⚙️  Após transformação:")
    print(f"   Registros inválidos removidos : {len(df_pedidos) - len(df_pedidos_t)}")
    print(f"   Pedidos válidos               : {len(df_pedidos_t)}")
    print(f"   Pedidos concluídos            : {len(df_ok)}")

    print(f"\n📥 Data Warehouse (loja_dw.db):")
    print(f"   dim_clientes  : {n_cli} registros")
    print(f"   dim_produtos  : {n_prod} registros")
    print(f"   fato_vendas   : {n_fat} registros")

    print(f"\n💰 Indicadores finais:")
    print(f"   Receita total : R$ {rec:>10,.2f}")
    print(f"   Lucro total   : R$ {luc:>10,.2f}")
    print(f"   Margem geral  : {luc/rec*100:.1f}%")
    print(f"\n{'='*55}")
    print("  🎉 Parabéns! Pipeline ETL concluído com sucesso!")
    print(f"{'='*55}")

except Exception as e:
    print(f"⚠️  Ainda há lacunas pendentes: {e}")
    print("   Complete todas as etapas anteriores e tente novamente.")


       RESUMO DO SEU PIPELINE ETL

📥 Fontes extraídas:
   Banco SQLite  → 10 clientes
   CSV           → 33 pedidos brutos
   Excel         → 8 produtos

⚙️  Após transformação:
   Registros inválidos removidos : 3
   Pedidos válidos               : 30
   Pedidos concluídos            : 22

📥 Data Warehouse (loja_dw.db):
   dim_clientes  : 10 registros
   dim_produtos  : 8 registros
   fato_vendas   : 22 registros

💰 Indicadores finais:
   Receita total : R$  62,664.37
   Lucro total   : R$  26,219.37
   Margem geral  : 41.8%

  🎉 Parabéns! Pipeline ETL concluído com sucesso!
